In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.5 MB/s eta 0:00:00


importing dataset

In [2]:
import zipfile, os

# Path to your zip in Google Drive — change the filename if different
zip_path = "/content/drive/MyDrive/Data_science_project/detection_dataset.zip"
extract_path = "/content/raw_dataset"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print("Extracted!")

Extracted!


 Merge all classes into one unified dataset with train/val split

In [3]:
import os, shutil, random

classes = ['crab', 'dolphin', 'octopus', 'shark', 'starfish', 'turtle']
splits = ['train', 'valid', 'test']
raw_root = "/content/raw_dataset/detection_dataset"
out_root = "/content/dataset"
val_ratio = 0.2
random.seed(42)

# Create output folders
for split in ['train', 'val']:
    os.makedirs(f"{out_root}/images/{split}", exist_ok=True)
    os.makedirs(f"{out_root}/labels/{split}", exist_ok=True)

def copy_files(img_src, lbl_src, img_dst, lbl_dst):
    images = [f for f in os.listdir(img_src) if f.endswith(('.jpg', '.png', '.jpeg'))]
    for img in images:
        stem = os.path.splitext(img)[0]
        lbl = stem + ".txt"
        if os.path.exists(os.path.join(lbl_src, lbl)):
            shutil.copy(os.path.join(img_src, img), os.path.join(img_dst, img))
            shutil.copy(os.path.join(lbl_src, lbl), os.path.join(lbl_dst, lbl))

for cls in classes:
    cls_path = os.path.join(raw_root, cls)
    all_images = []

    # Collect all images from all splits
    for split in splits:
        img_dir = os.path.join(cls_path, split, "images")
        lbl_dir = os.path.join(cls_path, split, "labels")
        if os.path.exists(img_dir):
            imgs = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
            for img in imgs:
                stem = os.path.splitext(img)[0]
                lbl = stem + ".txt"
                if os.path.exists(os.path.join(lbl_dir, lbl)):
                    all_images.append((os.path.join(img_dir, img), os.path.join(lbl_dir, lbl)))

    # Shuffle and split
    random.shuffle(all_images)
    val_count = max(1, int(len(all_images) * val_ratio))
    val_set = all_images[:val_count]
    train_set = all_images[val_count:]

    print(f"{cls}: {len(train_set)} train, {len(val_set)} val")

    for img_path, lbl_path in train_set:
        shutil.copy(img_path, f"{out_root}/images/train/{os.path.basename(img_path)}")
        shutil.copy(lbl_path, f"{out_root}/labels/train/{os.path.basename(lbl_path)}")

    for img_path, lbl_path in val_set:
        shutil.copy(img_path, f"{out_root}/images/val/{os.path.basename(img_path)}")
        shutil.copy(lbl_path, f"{out_root}/labels/val/{os.path.basename(lbl_path)}")

print("\nDone merging!")

crab: 358 train, 89 val
dolphin: 212 train, 52 val
octopus: 152 train, 38 val
shark: 397 train, 99 val
starfish: 160 train, 40 val
turtle: 160 train, 40 val

Done merging!


 Check label class indices and remap to 0–5

In [4]:
# Each roboflow dataset uses class index 0 for its own class
# We need to remap: crab=0, dolphin=1, octopus=2, shark=3, starfish=4, turtle=5

classes = ['crab', 'dolphin', 'octopus', 'shark', 'starfish', 'turtle']

def remap_labels(label_dir, class_name):
    class_id = classes.index(class_name)
    for lbl_file in os.listdir(label_dir):
        if not lbl_file.endswith(".txt"):
            continue
        path = os.path.join(label_dir, lbl_file)
        with open(path, 'r') as f:
            lines = f.readlines()
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if parts:
                parts[0] = str(class_id)  # remap class index
                new_lines.append(" ".join(parts))
        with open(path, 'w') as f:
            f.write("\n".join(new_lines))

# We need to track which label belongs to which class
# Re-do the merge with remapping built in
raw_root = "/content/raw_dataset/detection_dataset"
out_root = "/content/dataset"
random.seed(42)

# Clear and redo
shutil.rmtree(out_root)
for split in ['train', 'val']:
    os.makedirs(f"{out_root}/images/{split}", exist_ok=True)
    os.makedirs(f"{out_root}/labels/{split}", exist_ok=True)

for cls in classes:
    class_id = classes.index(cls)
    cls_path = os.path.join(raw_root, cls)
    all_images = []

    for split in ['train', 'valid', 'test']:
        img_dir = os.path.join(cls_path, split, "images")
        lbl_dir = os.path.join(cls_path, split, "labels")
        if os.path.exists(img_dir):
            imgs = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
            for img in imgs:
                stem = os.path.splitext(img)[0]
                lbl = stem + ".txt"
                if os.path.exists(os.path.join(lbl_dir, lbl)):
                    all_images.append((os.path.join(img_dir, img), os.path.join(lbl_dir, lbl)))

    random.shuffle(all_images)
    val_count = max(1, int(len(all_images) * 0.2))
    val_set = all_images[:val_count]
    train_set = all_images[val_count:]

    for dest_split, dataset in [('train', train_set), ('val', val_set)]:
        for img_path, lbl_path in dataset:
            img_name = os.path.basename(img_path)
            lbl_name = os.path.basename(lbl_path)

            shutil.copy(img_path, f"{out_root}/images/{dest_split}/{img_name}")

            # Remap label class index
            with open(lbl_path, 'r') as f:
                lines = f.readlines()
            new_lines = []
            for line in lines:
                parts = line.strip().split()
                if parts:
                    parts[0] = str(class_id)
                    new_lines.append(" ".join(parts))
            with open(f"{out_root}/labels/{dest_split}/{lbl_name}", 'w') as f:
                f.write("\n".join(new_lines))

    print(f"{cls} (id={class_id}): {len(train_set)} train, {len(val_set)} val")

print("\nAll done with remapping!")

crab (id=0): 358 train, 89 val
dolphin (id=1): 212 train, 52 val
octopus (id=2): 152 train, 38 val
shark (id=3): 397 train, 99 val
starfish (id=4): 160 train, 40 val
turtle (id=5): 160 train, 40 val

All done with remapping!


Create data.yaml

In [5]:
yaml_content = """path: /content/dataset
train: images/train
val: images/val

nc: 6
names: ['crab', 'dolphin', 'octopus', 'shark', 'starfish', 'turtle']
"""

with open("/content/dataset/data.yaml", "w") as f:
    f.write(yaml_content)

print("data.yaml created!")

data.yaml created!


train

In [6]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

results = model.train(
    data="/content/dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="aquatic_yolo",
    patience=10,
    save=True,
    plots=True
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgs

Evaluate and download

In [7]:
import shutil
from google.colab import files

metrics = model.val()
print(f"mAP50:    {metrics.box.map50:.2%}")
print(f"mAP50-95: {metrics.box.map:.2%}")

best = "/content/runs/detect/aquatic_yolo/weights/best.pt"
shutil.copy(best, "/content/drive/MyDrive/aquatic_yolo_best.pt")
files.download(best)

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1385.9±438.7 MB/s, size: 61.1 KB)
val: Scanning /content/dataset/labels/val.cache... 358 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 358/358 88.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.3s
                   all        358        419       0.91      0.875      0.923      0.699
                  crab         89         89      0.977          1      0.995      0.734
               dolphin         52         80      0.808      0.685      0.794      0.464
               octopus         38         40      0.889       0.95      0.968      0.882
                 shark         99        130      0.837      0.769      0.864      0.666
              starfish         40         40     

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>